# DFL 선수별 데드볼 판정

- **TranSPORTmer 참고** · 선수별 좌표 → 시간 attention 2블록 → 선수 간 attention 1블록을 두 단계로 반복 → 시점별 CLS → 데드볼 확률.
- **실험 범위** · DFL BallStatus 이진 분류. 공 좌표 복원과 라리가 학습본 재현은 별도 과제다. 공 좌표는 기존 라인아웃 후처리에만 사용한다.
- **입력** · 공개 DFL 변환 데이터 `data/external/dfl/*_{tracking,events,meta}.parquet`. 25fps에서 4프레임마다 한 번 샘플링해 9.6초·60시점 창을 만든다.
- **평가** · 6경기 학습·1경기 평가를 7번 반복한다. 기존 HSJ 노이즈와 비터비·라인아웃 보정을 사용한다. 모델 입력은 6.25Hz이며 검증은 하프별 선형 보간으로 복원한 25fps에서 계산한다.
- **환경** · 기존 dead_ball Python 의존성, SciPy, PyTorch 2.15. Apple Metal 또는 CPU.
- [TranSPORTmer 논문 Fig. 2](https://openaccess.thecvf.com/content/ACCV2024/papers/Capellera_TranSPORTmer_A_Holistic_Approach_to_Trajectory_Understanding_in_Multi-Agent_Sports_ACCV_2024_paper.pdf)

In [ ]:
import hashlib
from importlib.metadata import version
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/train_dfl.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
import compare_smoothing
from config import CONFIG
import dfl_labels
import dfl_load
import features
import final_dead_balls
import metrics
import simulate
import smoothing
import train_dfl

MATCHES = tuple(dfl_load.MATCHES)
FPS = CONFIG["fps"]
STEP = 4
WINDOW = 60
TRAIN_HOP = 60
VALID_HOP = 30
SLOTS = 26
WIDTH = 32
HEADS = 4
BATCH = 16
EPOCHS = 6
LR = 0.001
WEIGHT_DECAY = 0.001
SEED = 17
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(min(6, os.cpu_count() or 1))
RUN_ID = "dfl_transportmer_20261112_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
NOTEBOOK = ROOT / "model/영상추출/모델/dead_ball/dfl_transportmer.ipynb"
assert len(MATCHES) == 7 and FPS == 25 and WINDOW * STEP / FPS == 9.6
VITERBI = CONFIG["smooth_choice"]

## 선수별 입력

입력 특징은 피치 크기로 정규화한 x·y, 홈/원정 구분, 관측 여부다. DFL의 선수 ID를 창 안에서 유지한다. 교체로 한 창에 등장하는 선수가 22명을 넘으면 함께 유지하고, 빈 슬롯은 마스킹한다. 훈련과 검증에 기존 HSJ의 선수 누락·좌표 노이즈를 적용한다.

In [ ]:
def digest(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()


def halves(period):
    bounds = np.r_[0, np.flatnonzero(period[1:] != period[:-1]) + 1, len(period)]
    return [slice(a, b) for a, b in zip(bounds[:-1], bounds[1:])]


def pack_windows(match, label, hop):
    coordinates = match.players / np.asarray(match.pitch, dtype=np.float32) * 2 - 1
    period = np.asarray(match.period)
    index_parts, valid_parts = [], []
    for half in halves(period):
        rows = np.arange(half.start, half.stop, STEP)
        positions = np.arange(0, len(rows), hop)[:, None] + np.arange(WINDOW)[None, :]
        index_parts.append(rows[np.minimum(positions, len(rows) - 1)])
        valid_parts.append(positions < len(rows))
    indices = np.concatenate(index_parts)
    valid = np.concatenate(valid_parts)
    packed = np.zeros((len(indices), WINDOW, SLOTS, 4), dtype=np.float32)
    max_players = 0
    for i, (rows, real) in enumerate(zip(indices, valid)):
        window = coordinates[rows]
        seen = np.isfinite(window).all(axis=-1) & real[:, None]
        players = np.flatnonzero(seen.any(axis=0))
        count = len(players)
        assert count <= SLOTS, f"{count} distinct players exceeds {SLOTS}"
        max_players = max(max_players, count)
        observed = seen[:, players]
        packed[i, :, :count, :2] = np.where(observed[..., None], window[:, players], 0)
        packed[i, :, :count, 2] = match.team[players] * 2 - 1
        packed[i, :, :count, 3] = observed
    target = np.where(valid, label[indices], -1).astype(np.int8)
    assert np.isfinite(packed).all() and set(np.unique(target)) >= {0, 1}
    return dict(x=packed, valid=valid, indices=indices, target=target, max_players=max_players,
                visible_players=np.isfinite(match.players).all(axis=-1).sum(axis=-1))


def prepare_match(match_id, include_validation=True):
    tracking, events, meta = dfl_load.load_match(match_id)
    original = features.match_from_dfl(tracking, meta)
    label = dfl_labels.dead_label(tracking)
    seed = CONFIG["sim_seed"] + MATCHES.index(match_id)
    noisy_train = simulate.simulate(original, seed=seed,
                                    **train_dfl.random_settings(np.random.default_rng(seed)))
    training = pack_windows(noisy_train, label, TRAIN_HOP)
    if not include_validation:
        return training, None
    noisy_eval = simulate.simulate(original, seed=CONFIG["sim_seed"])
    evaluation = pack_windows(noisy_eval, label, VALID_HOP)
    evaluation.update(
        label=label, period=np.asarray(original.period), source_frame=np.asarray(original.frame),
        spells=dfl_labels.dead_spells(tracking, events),
        geometry=final_dead_balls.ball_signals(original),
    )
    evaluation["crossings"] = final_dead_balls.line_crossings(evaluation["geometry"])
    return training, evaluation


def move(batch, indices, include_target=False):
    x = torch.from_numpy(batch["x"][indices]).to(DEVICE)
    valid = torch.from_numpy(batch["valid"][indices]).to(DEVICE)
    if include_target:
        target = torch.from_numpy(batch["target"][indices]).to(DEVICE, dtype=torch.float32)
        return x, valid, target
    return x, valid

## 두 단계 시간·선수 간 attention

첫 단계의 시간 블록은 실제 관측값을 attention의 key로 사용한다. 둘째 단계는 첫 단계에서 만든 표현을 사용한다. 선수 간 블록은 존재하는 트랙과 CLS만 사용한다. 분류용 CLS는 시점마다 하나씩 둔다.

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(WIDTH)
        self.qkv = nn.Linear(WIDTH, 3 * WIDTH)
        self.out = nn.Linear(WIDTH, WIDTH)
        self.ffn_norm = nn.LayerNorm(WIDTH)
        self.ffn = nn.Sequential(nn.Linear(WIDTH, WIDTH * 2), nn.GELU(), nn.Linear(WIDTH * 2, WIDTH))

    def forward(self, value, keep):
        batch, length, _ = value.shape
        qkv = self.qkv(self.norm(value)).reshape(batch, length, 3, HEADS, WIDTH // HEADS)
        query, key, content = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(
            query.contiguous(), key.contiguous(), content.contiguous(),
            attn_mask=keep[:, None, None, :],
        ).transpose(1, 2).reshape(batch, length, WIDTH)
        value = value + self.out(context)
        return value + self.ffn(self.ffn_norm(value))


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal = nn.ModuleList([AttentionBlock(), AttentionBlock()])
        self.social = AttentionBlock()

    def forward(self, tokens, temporal_keep, social_keep):
        batch, steps, agents, _ = tokens.shape
        temporal = tokens.transpose(1, 2).reshape(batch * agents, steps, WIDTH)
        for block in self.temporal:
            temporal = block(temporal, temporal_keep.reshape(batch * agents, steps))
        social = temporal.reshape(batch, agents, steps, WIDTH).transpose(1, 2)
        social = social.reshape(batch * steps, agents, WIDTH)
        visible = social_keep[:, None].expand(batch, steps, agents).reshape(batch * steps, agents)
        return self.social(social, visible).reshape(batch, steps, agents, WIDTH)


class TransportmerDeadBall(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Sequential(nn.Linear(4, WIDTH), nn.GELU(), nn.Linear(WIDTH, WIDTH))
        self.cls = nn.Parameter(torch.zeros(1, 1, 1, WIDTH))
        self.coarse = Encoder()
        self.fine = Encoder()
        self.classifier = nn.Sequential(nn.LayerNorm(WIDTH), nn.Linear(WIDTH, 1))
        positions = torch.arange(WINDOW, dtype=torch.float32)[:, None]
        frequencies = 10000 ** (-torch.arange(0, WIDTH, 2, dtype=torch.float32) / WIDTH)
        phase = positions * frequencies[None, :]
        self.register_buffer("positional", torch.stack((phase.sin(), phase.cos()), -1).flatten(-2)[None, :, None])
        self.register_buffer("first", torch.arange(WINDOW) == 0)

    def forward(self, players, valid):
        batch, steps, _, _ = players.shape
        observed = (players[..., 3] > 0) & valid[:, :, None]
        agents = observed.any(dim=1)
        tokens = torch.cat((self.embedding(players), self.cls.expand(batch, steps, 1, WIDTH)), dim=2)
        coarse_keep = torch.cat((observed, valid[:, :, None]), dim=2).transpose(1, 2)
        coarse_keep = coarse_keep | (~coarse_keep.any(-1, keepdim=True) & self.first)
        social_keep = torch.cat((agents, torch.ones_like(agents[:, :1])), dim=1)
        fine_keep = valid[:, None, :].expand_as(coarse_keep)
        tokens = self.coarse(tokens + self.positional, coarse_keep, social_keep)
        tokens = self.fine(tokens + self.positional, fine_keep, social_keep)
        return self.classifier(tokens[:, :, -1]).squeeze(-1)


def synchronize():
    if DEVICE.type == "mps":
        torch.mps.synchronize()


def check_model():
    torch.manual_seed(SEED)
    model = TransportmerDeadBall().to(DEVICE).eval()
    with torch.inference_mode():
        x = torch.randn(2, WINDOW, SLOTS, 4, device=DEVICE)
        x[..., 3] = (x[..., 3] > 0).float()
        x[1, :, :, 3] = 0
        valid = torch.ones(2, WINDOW, dtype=torch.bool, device=DEVICE)
        original = model(x, valid).cpu()
        permuted = model(x.flip(dims=(2,)), valid).cpu()
    assert original.shape == (2, WINDOW) and torch.isfinite(original).all()
    torch.testing.assert_close(original, permuted, rtol=2e-4, atol=2e-4)
    return sum(parameter.numel() for parameter in model.parameters())

## 학습과 경기 단위 평가

학습은 6 epoch 고정, AdamW와 BCE. 각 epoch에 학습 6경기의 모든 창을 한 번씩 쓴다. 50% 겹치는 검증 창을 가중 평균하고, 25fps 프레임마다 확률과 데드볼 판정을 저장한다.

In [ ]:
def new_training():
    torch.manual_seed(SEED)
    model = TransportmerDeadBall().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, optimizer


def train_batch(model, optimizer, data, batch):
    x, valid, target = move(data, batch, include_target=True)
    optimizer.zero_grad(set_to_none=True)
    logits = model(x, valid)
    loss = (F.binary_cross_entropy_with_logits(logits, target.clamp_min(0), reduction="none") * valid).sum()
    loss = loss / valid.sum()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 5.0, foreach=False)
    optimizer.step()
    return loss.detach()


def benchmark(parameters):
    prepared = perf_counter()
    data, _ = prepare_match(MATCHES[0], include_validation=False)
    model, optimizer = new_training()
    for start in range(0, 3 * BATCH, BATCH):
        train_batch(model, optimizer, data, slice(start, start + BATCH))
    synchronize()
    started = perf_counter()
    for start in range(3 * BATCH, 15 * BATCH, BATCH):
        train_batch(model, optimizer, data, slice(start, start + BATCH))
    synchronize()
    per_batch = (perf_counter() - started) / 12
    estimated = per_batch * (len(data["x"]) * 6 / BATCH) * EPOCHS * 7
    print(f"{DEVICE} | {parameters:,} parameters | {per_batch:.3f}s/batch | "
          f"train estimate {estimated / 60:.1f}min | first preparation {started - prepared:.1f}s", flush=True)


def fit(matches, training):
    model, optimizer = new_training()
    data = {key: np.concatenate([training[m][key] for m in matches]) for key in ("x", "valid", "target")}
    rng = np.random.default_rng(SEED)
    history = []
    for epoch in range(1, EPOCHS + 1):
        started = perf_counter()
        order = rng.permutation(len(data["x"]))
        losses = [train_batch(model, optimizer, data, order[a:a+BATCH]) for a in range(0, len(order), BATCH)]
        loss = torch.stack(losses).mean().item()
        assert np.isfinite(loss)
        seconds = perf_counter() - started
        history.append(dict(epoch=epoch, train_bce=loss, seconds=seconds))
        print(f"  epoch {epoch}/{EPOCHS} | {seconds:.1f}s | BCE {loss:.4f}", flush=True)
    return model.eval(), history


def predict(model, evaluation):
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(evaluation["x"]), BATCH):
            x, valid = move(evaluation, slice(start, start + BATCH))
            probabilities.append(model(x, valid).sigmoid().cpu().numpy())
    prob = np.concatenate(probabilities)
    labels = evaluation["label"]
    numerator = np.zeros(len(labels), dtype=np.float64)
    denominator = np.zeros_like(numerator)
    valid = evaluation["valid"]
    indices = evaluation["indices"][valid]
    blend = 0.25 + np.sin(np.pi * (np.arange(WINDOW) + 0.5) / WINDOW) ** 2
    weights = np.broadcast_to(blend, valid.shape)[valid]
    np.add.at(numerator, indices, prob[valid] * weights)
    np.add.at(denominator, indices, weights)
    sampled = denominator > 0
    full = np.empty(len(labels), dtype=np.float64)
    for half in halves(evaluation["period"]):
        rows = np.arange(half.start, half.stop)
        sample = rows[sampled[half]]
        np.testing.assert_array_equal(sample, rows[::STEP])
        full[half] = np.interp(rows, sample, numerator[sample] / denominator[sample])
    assert np.isfinite(full).all() and np.all((0 <= full) & (full <= 1))
    return full, sampled


def score(evaluation, probability):
    label, period = evaluation["label"], evaluation["period"]
    frame = pd.DataFrame(dict(dead=label, period=period))
    free = np.zeros(len(label), dtype=bool)
    free[evaluation["crossings"]] = True
    viterbi = smoothing.viterbi(probability, period, VITERBI["penalty"], VITERBI["bias"])
    free_entry = smoothing.viterbi(probability, period, VITERBI["penalty"], VITERBI["bias"], free_entry=free)
    forced, _ = final_dead_balls.force_line_outs(free_entry, evaluation["crossings"],
                                                evaluation["geometry"], probability, period)
    rows = []
    for stage, state in (("viterbi", viterbi), ("line_out", forced)):
        segments = metrics.segment_metrics(evaluation["spells"], state, period)
        false_dead = sum(
            label[half][a:b].mean() < 0.5
            for half in halves(period) for a, b in features.runs_of(state[half] == 1)
        )
        rows.append(dict(stage=stage, **compare_smoothing.score_one(state, frame),
                         false_dead=int(false_dead), missed_dead=int((~segments["found"]).sum()),
                         start_abs_med_s=float(segments["start_abs_med"]),
                         end_abs_med_s=float(segments["end_abs_med"])))
    return rows, viterbi, forced


def write_frames(held, evaluation, prob, sampled, viterbi, forced):
    dest = OUT / "matches" / held
    dest.mkdir(parents=True, exist_ok=True)
    source = evaluation["source_frame"]
    frame = np.arange(len(source))
    assert len(np.unique(source)) == len(source)
    shared = dict(frame=frame, time_seconds=frame / FPS, source_frame_id=source)
    pd.DataFrame(dict(**shared, dead_score=prob, dead_viterbi=viterbi, dead=forced.astype(np.int8)))       .to_csv(dest / "features.csv", index=False)
    pd.DataFrame(dict(**shared, processed=1, model_sampled=sampled.astype(np.int8),
                      visible_players=evaluation["visible_players"])).to_csv(dest / "frame_status.csv", index=False)
    return [dest / "features.csv", dest / "frame_status.csv"]

## 실행

저장소 루트나 이 폴더에서 노트북 셀을 순서대로 실행한다. `MTAH_TRANSPORTMER_MODE=benchmark`는 첫 학습 경기에서 속도만 측정한다. 일반 실행은 7개 fold 결과·선수 모델 가중치·매니페스트를 같은 실행ID 폴더에 저장한다.

In [ ]:
def run_experiment(parameters):
    OUT.mkdir(parents=True, exist_ok=True)
    started = perf_counter()
    training, evaluations = {}, {}
    inputs = {}
    for match in MATCHES:
        for path in map(Path, dfl_load.cache_paths(match)):
            inputs[str(path.relative_to(ROOT))] = digest(path)
        training[match], evaluations[match] = prepare_match(match)
        print(f"준비 {match} | 학습 창 {len(training[match]['x'])} | "
              f"검증 {len(evaluations[match]['label']):,}프레임 | "
              f"창 내 최대 선수 {max(training[match]['max_players'], evaluations[match]['max_players'])}", flush=True)
    preparation_s = perf_counter() - started
    manifest_id = dict(notebook_sha256=digest(NOTEBOOK), input_sha256=inputs, device=str(DEVICE))
    checkpoint = OUT / "checkpoint.json"
    results = []
    if checkpoint.is_file():
        state = json.loads(checkpoint.read_text())
        assert state["fingerprint"] == manifest_id
        results = state["results"]
    by_match = {row["match"]: row for row in results}
    weights = OUT / "weights"
    weights.mkdir(exist_ok=True)

    for held in MATCHES:
        if held in by_match:
            continue
        started = perf_counter()
        matches = [m for m in MATCHES if m != held]
        assert len(matches) == 6 and held not in matches
        print(f"{held} 평가 | {', '.join(matches)} 학습", flush=True)
        fit_start = perf_counter()
        model, history = fit(matches, training)
        fit_s = perf_counter() - fit_start
        evaluation = evaluations[held]
        predict_start = perf_counter()
        probability, sampled = predict(model, evaluation)
        predict_s = perf_counter() - predict_start
        scores, viterbi, forced = score(evaluation, probability)
        exported = write_frames(held, evaluation, probability, sampled, viterbi, forced)
        weight_path = weights / f"{held}.pt"
        torch.save({name: value.cpu() for name, value in model.state_dict().items()}, weight_path)
        files = {str(path.relative_to(OUT)): digest(path) for path in [*exported, weight_path]}
        row = dict(match=held, train_matches=matches, frames=len(probability), sampled=int(sampled.sum()),
                   scores=scores, history=history, fit_s=fit_s, predict_s=predict_s,
                   fold_s=perf_counter() - started, file_sha256=files)
        results.append(row)
        pending = checkpoint.with_suffix(".tmp")
        pending.write_text(json.dumps(dict(fingerprint=manifest_id, results=results)), encoding="utf-8")
        pending.replace(checkpoint)
        print(f"{held} 완료 | {row['fold_s']:.1f}s | F1 "
              f"{scores[0]['alive_spell_f1']:.4f} / {scores[1]['alive_spell_f1']:.4f}", flush=True)

    per_match = pd.DataFrame([
        dict(match=fold["match"], fit_s=fold["fit_s"], predict_s=fold["predict_s"],
             fold_s=fold["fold_s"], **score_row)
        for fold in results for score_row in fold["scores"]
    ])
    assert len(per_match) == 14 and not per_match.duplicated(["match", "stage"]).any()
    assert per_match["alive_spell_f1"].between(0, 1).all()
    assert {fold["match"] for fold in results} == set(MATCHES)
    summary = (per_match.groupby("stage", sort=False)
               .agg(alive_spell_f1=("alive_spell_f1", "mean"), dead_frame_f1=("dead_f1", "mean"),
                    false_dead=("false_dead", "sum"), missed_dead=("missed_dead", "sum"),
                    start_abs_med_s=("start_abs_med_s", "mean"), end_abs_med_s=("end_abs_med_s", "mean"),
                    fit_s=("fit_s", "sum"), predict_s=("predict_s", "sum"), fold_s=("fold_s", "sum")).reset_index())
    summary.insert(0, "model", "TranSPORTmer-style")
    per_match.to_csv(OUT / "per_match.csv", index=False)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    pd.DataFrame([dict(match=fold["match"], **epoch) for fold in results for epoch in fold["history"]])       .to_csv(OUT / "training_history.csv", index=False)
    manifests = {name: digest(OUT / name) for name in ("per_match.csv", "model_summary.csv", "training_history.csv")}
    manifest = dict(
        run_id=RUN_ID, fingerprint=manifest_id, model_parameters=parameters,
        code_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        python=platform.python_version(), versions={name: version(name) for name in ("torch", "pandas", "pyarrow", "numpy")},
        model_config=dict(window=WINDOW, sample_step=STEP, batch=BATCH, epochs=EPOCHS, width=WIDTH, heads=HEADS,
                          encoders=2, temporal_blocks_per_encoder=2, social_blocks_per_encoder=1,
                          lr=LR, weight_decay=WEIGHT_DECAY, seed=SEED, train_hop=TRAIN_HOP, eval_hop=VALID_HOP),
        preparation_s=preparation_s, matches=results, report_sha256=manifests,
    )
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (OUT / "validation.json").write_text(json.dumps(dict(
        passed=True, held_matches=7, score_rows=14, frames=sum(fold["frames"] for fold in results),
        sampled_frames=sum(fold["sampled"] for fold in results), permutation_and_missing_smoke_test=True,
    ), indent=2) + "\n", encoding="utf-8")
    print(summary.round(4).to_string(index=False), flush=True)
    return summary


parameters = check_model()
print(f"device={DEVICE} | permutation & missing-coordinate check passed | parameters={parameters:,}", flush=True)
if os.environ.get("MTAH_TRANSPORTMER_MODE") == "benchmark":
    benchmark(parameters)
else:
    summary = run_experiment(parameters)